In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalog_name", "catalog_smartdata")
dbutils.widgets.text("gold_schema", "gold")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
gold_schema = dbutils.widgets.get("gold_schema")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {gold_schema}")


VISTA POWER BI - PRINCIPAL

In [0]:
from pyspark.sql import functions as F

In [0]:
fact = spark.table(f"{catalog_name}.{gold_schema}.fact_autos")
dim_make = spark.table(f"{catalog_name}.{gold_schema}.dim_make")
dim_body = spark.table(f"{catalog_name}.{gold_schema}.dim_body_style")
dim_engine = spark.table(f"{catalog_name}.{gold_schema}.dim_engine")
dim_fuel = spark.table(f"{catalog_name}.{gold_schema}.dim_fuel")
dim_drive = spark.table(f"{catalog_name}.{gold_schema}.dim_drive")

df = (
    fact.alias("f")
    .join(dim_make.alias("m"), "make_id")
    .join(dim_body.alias("b"), "body_style_id")
    .join(dim_engine.alias("e"), "engine_id")
    .join(dim_fuel.alias("fu"), "fuel_id")
    .join(dim_drive.alias("d"), "drive_id")
)

In [0]:
df_finance = (
    df
    .filter(F.col("price").isNotNull())
    .withColumn("price_per_hp", F.col("price") / F.col("horsepower"))
    .withColumn("price_per_engine", F.col("price") / F.col("engine_size"))
    .withColumn("avg_mpg", (F.col("city_mpg") + F.col("highway_mpg")) / 2)
)

In [0]:
from pyspark.sql import functions as F

fact = spark.table(f"{catalog_name}.{gold_schema}.fact_autos")
dim_make = spark.table(f"{catalog_name}.{gold_schema}.dim_make")
dim_body = spark.table(f"{catalog_name}.{gold_schema}.dim_body_style")
dim_engine = spark.table(f"{catalog_name}.{gold_schema}.dim_engine")
dim_fuel = spark.table(f"{catalog_name}.{gold_schema}.dim_fuel")
dim_drive = spark.table(f"{catalog_name}.{gold_schema}.dim_drive")

df = (
    fact.alias("f")
    .join(dim_make.alias("m"), "make_id")
    .join(dim_body.alias("b"), "body_style_id")
    .join(dim_engine.alias("e"), "engine_id")
    .join(dim_fuel.alias("fu"), "fuel_id")
    .join(dim_drive.alias("d"), "drive_id")
)

df_finance = (
    df
    .filter(F.col("price").isNotNull())
    .withColumn("price_per_hp", F.col("price") / F.col("horsepower"))
    .withColumn("price_per_engine", F.col("price") / F.col("engine_size"))
    .withColumn("avg_mpg", (F.col("city_mpg") + F.col("highway_mpg")) / 2)
)

df_finance.write.mode("overwrite").saveAsTable(f"{catalog_name}.{gold_schema}.vw_powerbi_auto_finance")

print("Vista vw_powerbi_auto_finance creada")

FINAL

In [0]:
from pyspark.sql.functions import avg, count

df_segments = (
    df
    .groupBy(
        "make",
        "body_style",
        "drive_wheels",
        "fuel_system"
    )
    .agg(
        count("*").alias("total_vehicles"),
        avg("price").alias("avg_price"),
        avg("horsepower").alias("avg_hp"),
        avg("city_mpg").alias("avg_city_mpg")
    )
)

df_segments.write.mode("overwrite").saveAsTable(f"{catalog_name}.{gold_schema}.vw_powerbi_auto_segments")

print("Vista vw_powerbi_auto_segments creada")